In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv')

In [12]:
df.head(20)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
5,events,manufacturing,1,59904.0,NaN,africa,6,0.83,1
6,social_media,technology,0,51283.0,NaN,middle_east,2,0.57,0
7,social_media,NaN,5,62975.0,student,europe,4,0.62,1
8,referral,healthcare,4,38648.0,unemployed,south_america,2,0.86,1
9,paid_ads,other,3,59866.0,student,australia,3,0.43,1


In [4]:
len(df)

1462

In [6]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [110]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['number_of_courses_viewed','annual_income','interaction_count','lead_score']
features = categorical + numerical

In [139]:
df[categorical].head()

,lead_source,industry,employment_status,location
0,paid_ads,NA,unemployed,south_america
1,social_media,retail,employed,south_america
2,events,healthcare,unemployed,australia
3,paid_ads,retail,NA,australia
4,referral,education,self_employed,europe


In [22]:
df[categorical].isnull().sum()

lead_source          128
industry             134
employment_status    100
location              63
dtype: int64

In [23]:
df[numerical].isnull().sum()

number_of_courses_viewed      0
annual_income               181
interaction_count             0
lead_score                    0
dtype: int64

In [28]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0)

In [29]:
df[categorical].isnull().sum()
df[numerical].isnull().sum()

number_of_courses_viewed    0
annual_income               0
interaction_count           0
lead_score                  0
dtype: int64

In [30]:
# Question 1

In [31]:
df['industry'].mode()

0    retail
Name: industry, dtype: object

In [145]:
df['industry'].value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

In [ ]:
# Question 2

In [32]:
df[numerical].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


In [146]:
df[numerical].corr().abs().stack().sort_values(ascending=False).head(16)

number_of_courses_viewed  number_of_courses_viewed    1.000000
annual_income             annual_income               1.000000
interaction_count         interaction_count           1.000000
lead_score                lead_score                  1.000000
annual_income             interaction_count           0.027036
interaction_count         annual_income               0.027036
number_of_courses_viewed  interaction_count           0.023565
interaction_count         number_of_courses_viewed    0.023565
annual_income             lead_score                  0.015610
lead_score                annual_income               0.015610
interaction_count         lead_score                  0.009888
lead_score                interaction_count           0.009888
number_of_courses_viewed  annual_income               0.009770
annual_income             number_of_courses_viewed    0.009770
number_of_courses_viewed  lead_score                  0.004879
lead_score                number_of_courses_viewed    0

In [ ]:
## annual_income and interaction_count

In [ ]:
## split the data

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

In [38]:
y_train = df_train.converted.values
y_val = df_val.converted.values

In [39]:
del df_train['converted']
del df_val['converted']

In [ ]:
# Question 3

In [45]:
from sklearn.metrics import mutual_info_score

def  calculate_mi(series):
    return mutual_info_score(series, df_train_full.converted)

In [49]:
df_mi = df_train_full[categorical].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')

display(round(df_mi.head(),2))

,MI
lead_source,0.03
employment_status,0.01
industry,0.01
location,0.00


In [50]:
# Question 4

In [136]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

train_dict = df_train[features].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

#model.predict_proba(X_val)
y_pred = model.predict_proba(X_val)[:, 1]
converted = y_pred > 0.5

accuracy = (y_val == converted).mean()

#accuracy = model.score(X_val, y_val)

print(accuracy)

0.6996587030716723


In [70]:
# Question 5

In [132]:
def train_without_feature(df_train, y_train, features):
    #display(features)
    train_dict = df_train[features].to_dict(orient='records')
    #display(train_dict)
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    
    model.fit(X_train, y_train)
    
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    model.predict_proba(X_val)
    y_pred = model.predict_proba(X_val)[:, 1]

    converted = y_pred > 0.5
    return (y_val == converted).mean()

#print(features)
    
accuracies_removed = {}
for feature in features:
    features_tmp = features.copy()
    features_tmp.remove(feature)
    #print(features_tmp)
    accuracies_removed[feature] = train_without_feature(df_train, y_train, list(features_tmp))

#print(accuracies_removed)

for k,v in accuracies_removed.items():
    print(f"{k:<25} {v:^10} {accuracy-v:>20}")

lead_source               0.7030716723549488 -0.0034129692832765013
industry                  0.6996587030716723                  0.0
employment_status         0.6962457337883959 0.0034129692832763903
location                  0.7098976109215017 -0.010238907849829393
number_of_courses_viewed  0.5563139931740614  0.14334470989761094
annual_income             0.8532423208191127 -0.15358361774744034
interaction_count         0.5563139931740614  0.14334470989761094
lead_score                0.7064846416382252 -0.0068259385665528916


In [127]:
## 'industry' has the smallest diff

In [128]:
# Question 6

In [129]:
for reg_strength in [0.01, 0.1, 1, 10, 100]:
    train_dict = df_train[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(solver='liblinear', C=reg_strength, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model.predict_proba(X_val)
    y_pred = model.predict_proba(X_val)[:, 1]

    converted = y_pred > 0.5

    print(reg_strength, (y_val == converted).mean())

0.01 0.6996587030716723
0.1 0.6996587030716723
1 0.6996587030716723
10 0.6996587030716723
100 0.6996587030716723
